### Key Findings from EDA

- 3 core GSR elements (PbBaSb), likely the strongest indicator
- Elemental ratios may yield discriminative signal that raw concentrations do not
- Barium = problematic for false positives --> GSR/Non-GSR boundary overlapping (BaCaSi, BaAl, BaSb, PbBa)
- Non-Barium confounders = Ca, Si, Al
- Based on EDA, in some cases oxygen may dilute particle composition (experiment with & w/o oxygen for some features)
- CuZn highly correlated Non-GSR

In [165]:
import pandas as pd
import numpy as np

# Using "average precision score" to estimate PR-AUC for standalone feature precision
from sklearn.metrics import average_precision_score as ap_score

In [166]:
element_cols = [
    'o', 's', 'cu', 'ba', 'al', 'si', 'ca', 'pb', 'sb', 'fe',
    'zn', 'cl', 'k', 'na', 'mg', 'ti', 'sn', 'p', 'mn', 'as',
    'cr', 'br', 'mo', 'sr', 'ni', 'w', 'hg'
]

In [167]:
df = pd.read_parquet("../../../data/processed/preprocessed.parquet")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")

new_feats = pd.DataFrame()

Shape: (2294985, 34)
Columns: Index(['stub_id', 'particle_id', 'relevance_class', 'merged_relevance_class',
       'final_class', 'label', 'o', 's', 'cu', 'ba', 'al', 'si', 'ca', 'pb',
       'sb', 'fe', 'zn', 'cl', 'k', 'na', 'mg', 'ti', 'sn', 'p', 'mn', 'as',
       'cr', 'br', 'mo', 'sr', 'ni', 'w', 'hg', 'target'],
      dtype='str')


In [168]:
# Estimating PR-AUC with average_precision_score
def compute_prauc(vals):
    y = (df["target"] == 1).astype(int).values # GSR
    return round(ap_score(y, vals), 4)

-----
## Feature: 3 Core GSR Elements --> Count

A `core_gsr_count` feature will check whether Pb, Ba, and Sb exist in the particle. The existance of 1 of those elements counts as 1, so if all 3 exist in the particle the count will be 3.

Based on EDA, we expect that nearly every "core_gsr_count = 3" will be GSR, and in contrast every "core_gsr_count = 0" will be Non-GSR.

In [169]:
new_feats['core_gsr_count'] = (
    (df['pb'] > 0).astype(int) + (df['ba'] > 0).astype(int) + (df['sb'] > 0).astype(int)
)

# should be 3
print(f"Highest count of core GSR elements in a particle: {new_feats['core_gsr_count'].max()}")
# Estimate PR-AUC
print(f"PR-AUC estimate for core_gsr_count: {compute_prauc(new_feats['core_gsr_count'].values)}")

Highest count of core GSR elements in a particle: 3
PR-AUC estimate for core_gsr_count: 0.9976


In [170]:
# Matrix to view GSR vs Non-GSR distribution across Core GSR Counts
pd.crosstab(df['label'], new_feats['core_gsr_count'], margins=True)

core_gsr_count,0,1,2,3,All
label,,,,,
GSR,0,0,536881,542065,1078946
Non_GSR,484704,726207,5092,36,1216039
All,484704,726207,541973,542101,2294985


Concern for false-positives regarding the 36 non-GSR particles that have all 3 core GSR elements present. Need to engineer additional feature(s) based on indentifying these 36 as non-GSR.

In [171]:
# What is the elemental composition of these 36 particles?
subset = df[(new_feats['core_gsr_count'] == 3) & (df['label'] == 'Non_GSR')]
medians = subset[element_cols].median()
medians[medians > 0].sort_values(ascending=False).to_frame('median')

,median
o,46.182249
ba,22.044205
al,4.948296
pb,0.956436
sb,0.771331
s,0.722646
cu,0.635611
si,0.507025
ca,0.309733


This makes sense. Most cases appear to be high barium with trace amounts of Pb and Sb (possible contamination; not enough of a presence to consider them as "core").

The `core_gsr_count` seems to be incredibly accurate at identifying GSR particles when all 3 core elements are present. The edge cases are when trace amounts of lead and antimony are present in a Non-GSR barium-heavy particle. The core GSR count feature fails here because it doesn't factor how much of the elements are present, just whether they exist.

In [172]:
# Class distribution of the 36 particles
df[(new_feats['core_gsr_count'] == 3) & (df['label'] == 'Non_GSR')]['final_class'].value_counts()

final_class
BaAl      30
CuZn       5
BaCaSi     1
Name: count, dtype: int64

31 of the 36 particles are barium-heavy classes. 
The other 5 are brass particles (heavy copper + zinc).

In [173]:
cols = ['final_class', 'label', 'pb', 'sb', 'ba', 'al', 'cu', 'zn', 'ca', 'si']
df[(new_feats['core_gsr_count'] == 3) & (df['label'] == 'Non_GSR')][cols]

,final_class,label,pb,sb,ba,al,cu,zn,ca,si
24650,BaAl,Non_GSR,1.339027,0.835819,5.061492,1.136361,35.421410,0.000000,0.320336,5.259798
122949,BaAl,Non_GSR,0.900112,0.522460,5.093280,32.028179,0.000000,0.412590,0.696888,2.277418
123329,BaAl,Non_GSR,0.987811,0.565076,4.944564,31.917309,0.000000,0.467532,0.459105,2.614125
269164,BaAl,Non_GSR,0.924694,0.975342,43.079071,18.979769,0.618005,0.000000,0.973265,0.000000
269661,BaAl,Non_GSR,0.988405,0.747169,37.029652,23.325136,0.000000,0.000000,0.674974,0.000000
288799,BaAl,Non_GSR,0.945907,0.787318,26.803625,28.013432,0.433905,0.018190,0.774001,0.000000
289101,BaAl,Non_GSR,0.974151,0.779309,33.954227,25.516876,0.512154,0.000000,0.423496,0.000000
290714,BaAl,Non_GSR,0.997457,0.643988,20.515358,23.227581,0.356044,0.133525,0.433891,0.391882
290797,BaAl,Non_GSR,0.937772,0.902222,28.305021,22.315155,0.000000,0.000000,0.556918,0.000000
295620,BaAl,Non_GSR,0.841732,0.242981,24.887445,27.260403,0.534635,0.000000,0.000000,0.644959


Need a feature that quantifies Pb and Sb to mitigate edge cases (most showing trace amounts of Pb and Sb).

I believe multiplicative quantification is best here for measuring Pb and Sb to address these 3 scenarios:
- Large quantities of both are weighted higher
- Trace quantities of both are penalized strongly
- A large quantity of 1 and trace quantity of 1 get slightly penalized

The reason for penalizing the 3rd scenario, instead of additive (Pb + Sb) is because a trace amount of any of these elements should be taken with caution due to potential contamination and false-positives as seen above. The specific example that catches my eye is row 1611349 where Pb=11.2 and Sb=0.9 ...  It would be best to handle this as multiplicative and not additive considering this is Non-GSR and we don't want to increase the weight of Pb & Sb's presence.

-----
## Feature: Pb x Sb (multiplicative quantification)

Based on the edge cases from the Core GSR Count feature, this "Pb x Sb" feature will be complimentary by quantifying Pb and Sb, and penalizing trace amounts to mitigate false-positives in some of the brass and barium-heavy Non-GSR particles.

In [174]:
new_feats['pb_times_sb'] = df['pb'] * (df['sb'])

# Estimate PR-AUC
print(f"PR-AUC estimate for pb_times_sb: {compute_prauc(new_feats['pb_times_sb'].values)}")

PR-AUC estimate for pb_times_sb: 0.8323


In [175]:
# What do the values look like for the 36 Non-GSR particles?
mask_36 = (new_feats['core_gsr_count'] == 3) & (df['label'] == 'Non_GSR')
new_feats['pb_times_sb'][mask_36].sort_values(ascending=False).to_frame()

,pb_times_sb
1611349,10.240610
1602467,4.122087
1372501,3.936304
2028013,3.303857
1603464,2.124097
884072,1.450204
1951925,1.348759
24650,1.119184
1479908,0.930282
884581,0.909322


Overall the `pb_times_sb` feature looks like it could be useful for mitigating these edge cases. The one concern may be that row 1611349 that I mentioned earlier.

__Possible consideration__: Find a feature that highlights the presence of brass particles (copper & zinc) to see if that might be useful for edge cases.

-----
## Feature(s): Elemental Ratios

### NOTE: NOT USING THIS FEATURE (using next feature in place of this)

Experimentation with various elemental ratios may identify relationships with discriminative signal that individual element composition cannot.

Initial ratio features based on EDA include:
- Pb / Ba
- Sb / Ba
- (Pb + Sb) / Ba
- Pb / Sb
- Cu / Zn (or Zn / Cu ?)

The first 3 directly address the complex relationship with Barium as an overlapping GSR/Non-GSR element. 

Pb / Sb is a measure of the Pb-Sb balance. 

The Cu & Zu relationship may be useful for addressing the edge cases with brass particles that have trace amounts of lead and antimony (as seen in the previous feature analysis).

__Pb / Ba__

In [176]:
# Pb / Ba
new_feats['pb_over_ba'] = df['pb'] / (df['ba'])

# PR-AUC estimate
# print(f"PR-AUC estimate for pb_over_ba: {compute_prauc(new_feats['pb_over_ba'].values)}")

# FAILED DUE TO NaN ...
# For div by zero, also need to check for infinity
problem_mask = np.isinf(new_feats['pb_over_ba']) | new_feats['pb_over_ba'].isna()
problem_df = df[problem_mask][['pb', 'ba']].assign(
    result=new_feats['pb_over_ba'][problem_mask].replace([np.inf, -np.inf], 'inf')
)
# Check div by zero (inf) vs. 0/0 (NaN)
inf_rows = problem_df[problem_df['result'] == 'inf']
nan_rows = problem_df[problem_df['result'].isna()]

print(f"inf rows (total: {len(inf_rows):,})")
print(inf_rows.head(5))
print(f"\nNaN rows (total: {len(nan_rows):,})")
print(nan_rows.head(5))

inf rows (total: 196,941)
           pb   ba result
0   12.783856  0.0    inf
4    8.541641  0.0    inf
17  35.978222  0.0    inf
19   9.523825  0.0    inf
21   9.484592  0.0    inf

NaN rows (total: 492,073)
      pb   ba result
707  0.0  0.0    NaN
709  0.0  0.0    NaN
710  0.0  0.0    NaN
712  0.0  0.0    NaN
713  0.0  0.0    NaN


'0/0' could be replaced by 0, but when numerator is > 0 this would not be an appropriate resolution.



### Handling 'inf'' for ratios where Barium = 0 and numerator > 0

__Epsilon approach__

In instances where Barium is 0 it will be subsituted with a very small decimal to avoid Inf.

This still resolves NaN to 0, because if the numerator is 0 the division result will be 0.

In [177]:
def safe_divide_with_eps(n, d):
    e = 1e-6
    safe_d = np.where(d == 0, e, d)
    return n / safe_d

In [178]:
new_feats['pb_over_ba'] = safe_divide_with_eps(df['pb'].values, df['ba'].values)

# confirm no 'inf' or 'nan'
any(np.isinf(new_feats['pb_over_ba']) | new_feats['pb_over_ba'].isna())

False

In [179]:
# estimate pr-auc
print(f"PR-AUC estimate for pb_over_ba: {compute_prauc(new_feats['pb_over_ba'].values)}")

PR-AUC estimate for pb_over_ba: 0.9443


In [180]:
# sanity check the vals
print(f"Min pb over ba = {new_feats['pb_over_ba'].min()}")
print(f"Max pb over ba = {new_feats['pb_over_ba'].max()}")

Min pb over ba = 0.0
Max pb over ba = 95775161.74316406


#### Potential bias with epsilon approach for handling 'inf' ... didn't account for how much it would inflate the ratio.... need to review if this approach is appropriate.

Another thing to consider here is that in the full dataset (which includes ambiguous labels) if Pb>0 and Ba=0,  this scenario could be where Pb is the only core GSR element present, in which case this ratio may not be a value that the model should use to classify the label.

Therefore, an alternative approach might be to use a sentinel value that signals Ba=0 (or any denominator as 0) to be deemed unreliable. For example, when __d=0__ replace it with __-1__ so that any `inf` scenario outputs a negative quotient. The hope would be that a tree model like xgboost would identify the threshold between positive and negative numbers, interpreting positive quotients as reliable signals and negative quotients as unreliable and instead would rely on other features. Or perhaps the sentinel value is only applied if `gsr_core_count=1` but would use an epsilon value if Pb and Sb both exist (gsr_core_count=2). However, we'd still need to determine whether an epsilon value would be introducing any bias.

__*Skipping single-element denominator ratios until an appropriate 'inf' strategy is determined*__

In [181]:
# Remove elemental ratios for now
new_feats.drop(columns=['pb_over_ba'], inplace=True)

new_feats.columns

Index(['core_gsr_count', 'pb_times_sb'], dtype='str')

_____
## Features for 'Core GSR Elements = 2'

From the distribution matrix for the Core GSR Count feature, there were 5,092 Non-GSR particles with `core_gsr_count=2`. We'll need some features to help distinguish these from GSR to minimize false-positives.

In [182]:
# Class distribution of the 5,092 particles
df[(new_feats['core_gsr_count'] == 2) & (df['label'] == 'Non_GSR')].groupby('final_class').agg(
    count=('pb', 'size'),
    pb_median=('pb', 'median'),
    pb_max=('pb', 'max'),
    sb_median=('sb', 'median'),
    sb_max=('sb', 'max'),
).sort_values('count', ascending=False)

,count,pb_median,pb_max,sb_median,sb_max
final_class,,,,,
BaAl,4077,0.000000,11.620400,0.803285,1.978046
BaCaSi,661,0.942170,7.834150,0.000000,0.995508
CuZn,298,0.000000,21.747881,1.624879,3.569439
Hg,35,0.760992,2.336872,20.959925,56.606514
TiZnGd,18,17.198016,73.943672,3.270193,11.296607
ZnTi,3,7.850978,9.904734,0.961876,0.971031


Barium-heavy classes BaAl and BaCaSi make up the vast majority of these cases and look like similar issues as the previously analyzed 36 Non-GSR particles. Mostly trace amounts of Pb and/or Sb contaminating the particles. 

1 notable difference is Pb median of 0 fo BaAi and Sb median of 0 for BaCaSi, which is due to particles that only have 1 or the other (Pb or Sb) but not both. This may impact the effectiveness of the previously engineered feature `pb_times_sb` because for all the GSR labeled particles with only PbBa or BaSb, the `pb_times_sb` will be 0 and therefore provides no useful information for the model. __New ratio features will need to be introduced, where additive quantification may be the better option than multiplicative__

The CuZn class has a little more concern that the Barium-heavy classes. Its max Pb val is 21.7 (not a trace amount). __Need feature to address CuZn levels to distinguish brass particles from GSR__

The 3 classes that didn't appear for the 36 Non-GSR particle analysis earlier are
- Hg
- TiZnGd
- ZnTi 

... making up just 56 particles, but clearly having significantly more than trace amounts of lead and antimony. __Investigate features regarding Hg, Ti, Zn, and Gd levels__

In [183]:
# Hg / Ti / Zn / Cu

# All particles
df.groupby(['final_class', 'label']).agg(
    hg_median=('hg', 'median'), hg_min=('hg', 'min'), hg_max=('hg', 'max'),
    ti_median=('ti', 'median'), ti_min=('ti', 'min'), ti_max=('ti', 'max'),
    zn_median=('zn', 'median'), zn_min=('zn', 'min'), zn_max=('zn', 'max'),
    cu_median=('cu', 'median'), cu_min=('cu', 'min'), cu_max=('cu', 'max'),
).sort_values('final_class')

,,hg_median,hg_min,hg_max,ti_median,ti_min,ti_max,zn_median,zn_min,zn_max,cu_median,cu_min,cu_max
final_class,label,,,,,,,,,,,,
BaAl,Non_GSR,0.000000,0.000000,4.884712,0.000000,0.000000,32.464085,0.000000,0.000000,87.366043,0.000000,0.000000,87.685516
BaCaSi,Non_GSR,0.000000,0.000000,3.839796,0.000000,0.000000,34.486923,0.000000,0.000000,79.046120,0.000000,0.000000,85.544052
BaSb,GSR,0.000000,0.000000,20.446365,0.000000,0.000000,23.568598,0.000000,0.000000,75.084702,1.803095,0.000000,87.667999
CuZn,Non_GSR,0.000000,0.000000,39.400715,0.000000,0.000000,0.999826,4.485462,1.000004,98.960793,5.440648,1.000018,96.011391
GaCuSn,Non_GSR,0.000000,0.000000,0.000000,0.000000,0.000000,36.994362,0.000000,0.000000,48.396103,45.880486,1.017354,93.534691
Hg,Non_GSR,4.756312,1.000075,81.537910,0.000000,0.000000,19.962116,0.000000,0.000000,45.564110,0.390263,0.000000,76.267967
PbBa,GSR,0.000000,0.000000,4.376632,0.000000,0.000000,28.563610,0.000000,0.000000,87.140335,1.224619,0.000000,86.915276
PbBaSb,GSR,0.000000,0.000000,15.893229,0.000000,0.000000,16.536961,0.000000,0.000000,72.920502,2.621746,0.000000,88.638824
PbSb,GSR,0.000000,0.000000,35.197956,0.000000,0.000000,49.475693,0.000000,0.000000,71.253693,1.446411,0.000000,92.481079


In [184]:
# Hg / Ti / Zn / Cu

# Only the 5,092 Non-GSR particles with Core GSR Count = 2
mask_5092 = (new_feats['core_gsr_count'] == 2) & (df['label'] == 'Non_GSR')
df[mask_5092].groupby(['final_class', 'label']).agg(
    hg_median=('hg', 'median'), hg_min=('hg', 'min'), hg_max=('hg', 'max'),
    ti_median=('ti', 'median'), ti_min=('ti', 'min'), ti_max=('ti', 'max'),
    zn_median=('zn', 'median'), zn_min=('zn', 'min'), zn_max=('zn', 'max'),
    cu_median=('cu', 'median'), cu_min=('cu', 'min'), cu_max=('cu', 'max'),
).sort_values('final_class')

,,hg_median,hg_min,hg_max,ti_median,ti_min,ti_max,zn_median,zn_min,zn_max,cu_median,cu_min,cu_max
final_class,label,,,,,,,,,,,,
BaAl,Non_GSR,0.000000,0.000000,2.988129,0.000000,0.000000,16.359972,0.000000,0.000000,87.366043,0.924578,0.000000,77.047401
BaCaSi,Non_GSR,0.000000,0.000000,1.709301,0.000000,0.000000,16.536423,0.000000,0.000000,76.919266,0.745724,0.000000,60.997234
CuZn,Non_GSR,0.000000,0.000000,0.000000,0.000000,0.000000,0.462312,1.941789,1.011265,29.231245,3.608578,1.001423,78.163727
Hg,Non_GSR,5.273128,1.105667,23.495045,0.000000,0.000000,0.000000,0.000000,0.000000,7.953229,3.499879,0.000000,21.721394
TiZnGd,Non_GSR,0.000000,0.000000,0.000000,3.857289,1.448742,9.888797,3.759074,1.555853,23.504723,5.217032,0.000000,23.918617
ZnTi,Non_GSR,0.000000,0.000000,0.000000,1.566514,1.240302,3.991877,27.297768,2.786701,55.709164,0.656709,0.626600,0.718531


__Hg__: This looks distinguishable simply by the raw element quantity. No new engineered features yet. The Hg class is very distinct.

__Ti & Zn__: An additive ratio over total mass might be useful for Ti and Zn

__Cu & Zn__: Clear benefit to having an additive ratio over total mass

__Gd__: This element didn't make the cut for 27 informative elements. Depending on the models ability to catch this small number of Non-GSR particles, we may revisit including this element. The (Ti+Zn)/Mass feature may be able to separate TiZnGd Non-GSR particles sufficiently.



### Measuring Percentage of Total Mass:

#### An alternative approach to Element Ratios

An alternative to ratios between direct elements, which has the potential downside of a 0 denominator, we can check the ratio between GSR elements over the total mass of the particle.

- (Pb+Ba) / (total mass - Sb)
- (Pb+Sb) / (total mass - Ba)
- (Ba+Sb) / (total mass - Pb)
- (Cu+Zn) / (total mass)
- (Ti+Zn) / (total mass)

I'm also curious to see if oxygen dilutes ratios at all, so will look at:

- (Pb+Ba) / (total mass - Sb - O)
- (Pb+Sb) / (total mass - Ba - O)
- (Ba+Sb) / (total mass - Pb - O)
- (Cu+Zn) / (total mass - O)
- (Ti+Zn) / (total mass - O)

The inclusion of Cu & Zn is for the Non-GSR brass particles that have core GSR elements present.

The inclusion of Ti & Zn is for the Non-GSR TiZnGd & TiZn Non-GSR glasses with core GSR count of 2.

__Calculate Mass (denominator)__

In [185]:
total_mass = df[element_cols].sum(axis=1)
total_mass_no_o = df[element_cols].sum(axis=1) - df['o']
# PbBa numerator
total_mass_no_sb = df[element_cols].sum(axis=1) - df['sb']
total_mass_no_sb_o = df[element_cols].sum(axis=1) - df['sb'] - df['o']
# PbSb numerator
total_mass_no_ba = df[element_cols].sum(axis=1) - df['ba']
total_mass_no_ba_o = df[element_cols].sum(axis=1) - df['ba'] - df['o']
# BaSb numerator
total_mass_no_pb = df[element_cols].sum(axis=1) - df['pb']
total_mass_no_pb_o = df[element_cols].sum(axis=1) - df['pb'] - df['o']


__Pb+Ba / (mass - Sb)__ and __Pb+Ba / (mass - Sb - O)__

In [186]:
new_feats['pb_ba_over_non_sb_mass'] =  (df['pb'] + df['ba']) / total_mass_no_sb
new_feats['pb_ba_over_non_sb_o_mass'] =  (df['pb'] + df['ba']) / total_mass_no_sb_o

# estimate pr-auc
print(f"PR-AUC estimate for pb_ba_over_non_sb_mass: {compute_prauc(new_feats['pb_ba_over_non_sb_mass'].values)}")
print(f"PR-AUC estimate for pb_ba_over_non_sb_o_mass: {compute_prauc(new_feats['pb_ba_over_non_sb_o_mass'].values)}")

PR-AUC estimate for pb_ba_over_non_sb_mass: 0.6925
PR-AUC estimate for pb_ba_over_non_sb_o_mass: 0.7736


__Pb+Sb / (mass - Ba)__ and __Pb+Sb / (mass - Ba - O)__

In [187]:
new_feats['pb_sb_over_non_ba_mass'] =  (df['pb'] + df['sb']) / total_mass_no_ba
new_feats['pb_sb_over_non_ba_o_mass'] =  (df['pb'] + df['sb']) / total_mass_no_ba_o

# estimate pr-auc
print(f"PR-AUC estimate for pb_sb_over_non_ba_mass: {compute_prauc(new_feats['pb_sb_over_non_ba_mass'].values)}")
print(f"PR-AUC estimate for pb_sb_over_non_ba_o_mass: {compute_prauc(new_feats['pb_sb_over_non_ba_o_mass'].values)}")

PR-AUC estimate for pb_sb_over_non_ba_mass: 0.9979
PR-AUC estimate for pb_sb_over_non_ba_o_mass: 0.9988


__Ba+Sb / (mass - Pb)__ and __Ba+Sb / (mass - Pb - O)__

In [188]:
new_feats['ba_sb_over_non_pb_mass'] =  (df['ba'] + df['sb']) / total_mass_no_pb
new_feats['ba_sb_over_non_pb_o_mass'] =  (df['ba'] + df['sb']) / total_mass_no_pb_o

# estimate pr-auc
print(f"PR-AUC estimate for ba_sb_over_non_pb_mass: {compute_prauc(new_feats['ba_sb_over_non_pb_mass'].values)}")
print(f"PR-AUC estimate for ba_sb_over_non_pb_o_mass: {compute_prauc(new_feats['ba_sb_over_non_pb_o_mass'].values)}")

PR-AUC estimate for ba_sb_over_non_pb_mass: 0.6172
PR-AUC estimate for ba_sb_over_non_pb_o_mass: 0.6914


__Cu+Zn / (mass)__ and __Cu+Zn / (mass - O)__

In [189]:
new_feats['cu_zn_over_mass'] =  (df['cu'] + df['zn']) / total_mass
new_feats['cu_zn_over_non_o_mass'] =  (df['cu'] + df['zn']) / total_mass_no_o

# estimate pr-auc
print(f"PR-AUC estimate for cu_zn_over_mass: {compute_prauc(new_feats['cu_zn_over_mass'].values)}")
print(f"PR-AUC estimate for cu_zn_over_non_o_mass: {compute_prauc(new_feats['cu_zn_over_non_o_mass'].values)}")

PR-AUC estimate for cu_zn_over_mass: 0.4329
PR-AUC estimate for cu_zn_over_non_o_mass: 0.4298


__Ti+Zn / (mass)__ and __Ti+Zn / (mass - O)__

In [190]:
new_feats['ti_zn_over_mass'] =  (df['ti'] + df['zn']) / total_mass
new_feats['ti_zn_over_non_o_mass'] =  (df['ti'] + df['zn']) / total_mass_no_o

# estimate pr-auc
print(f"PR-AUC estimate for ti_zn_over_mass: {compute_prauc(new_feats['ti_zn_over_mass'].values)}")
print(f"PR-AUC estimate for ti_zn_over_non_o_mass: {compute_prauc(new_feats['ti_zn_over_non_o_mass'].values)}")

PR-AUC estimate for ti_zn_over_mass: 0.401
PR-AUC estimate for ti_zn_over_non_o_mass: 0.4004


## Feature: Alternative Additive for Pb + Sb

A key GSR indicator that separates Ba and considers when only 2 core GSR elements exist is the (Pb+Sb)/(mass - Ba) feature. This specifically looks to assist in the false-positive situations where Pb and/or Sb were present but in small quantities, as identified in previous sections.

An alternative to this is a logarithmic approach, to separate the lower Pb & Sb quantities from the higher ones with more distinction, while still using additive quantification to allow for the scenario where only Pb or Sb might exist, but not both.

In [191]:
new_feats['log_pb_plus_sb'] = np.log1p(df['pb'] + df['sb'])

# estimate pr-auc
print(f"PR-AUC estimate for log_pb_plus_sb: {compute_prauc(new_feats['log_pb_plus_sb'].values)}")

PR-AUC estimate for log_pb_plus_sb: 0.9973


_____
## Feature: Environmental & Mineral Confounders

### (domain knowledge)

Based on a combination of EDA, feature exploration, and domain knowledge, the top GSR confounders include:
- Ca
- Si
- Al
- Fe
- Ti
- Zn
- Cu

These are often found in things such as fireworks, brakepads, industrial materials, and environmental particles.

This feature will calculate the ratio of GSR element mass over the confounder element mass, but specifically removing barium since it is heavily present in both GSR and Non-GSR particles.

In [192]:
gsr = df['pb'] + df['sb']
confounders = df['ca'] + df['si'] + df['al'] + df['fe'] + df['ti'] + df['zn'] + df['cu']
new_feats['gsr_over_confounders'] = gsr / confounders

# estimate pr-auc
# print(f"PR-AUC estimate for gsr_over_confounders: {compute_prauc(new_feats['gsr_over_confounders'].values)}")

# PROBLEMATIC ZERO DENOMINATOR AGAIN ... Arggghh


problem_mask = np.isinf(new_feats['gsr_over_confounders']) | new_feats['gsr_over_confounders'].isna()
problem_df = df[problem_mask][['pb', 'sb']].assign(
    gsr=gsr[problem_mask],
    confounders=confounders[problem_mask],
    result=new_feats['gsr_over_confounders'][problem_mask].replace([np.inf, -np.inf], 'inf')
)

inf_rows = problem_df[problem_df['result'] == 'inf']
nan_rows = problem_df[problem_df['result'].isna()]

print(f"inf rows (total: {len(inf_rows):,})")
print(inf_rows.head(5))
print(f"\nNaN rows (total: {len(nan_rows):,})")
print(nan_rows.head(5))



inf rows (total: 108,183)
           pb         sb        gsr  confounders result
0   12.783856  51.773666  64.557523          0.0    inf
1   21.533052   1.546292  23.079345          0.0    inf
4    8.541641  26.754719  35.296360          0.0    inf
7   13.361807  16.949888  30.311695          0.0    inf
12  19.382690  29.875378  49.258068          0.0    inf

NaN rows (total: 191)
        pb   sb  gsr  confounders result
4737   0.0  0.0  0.0          0.0    NaN
38047  0.0  0.0  0.0          0.0    NaN
58132  0.0  0.0  0.0          0.0    NaN
58251  0.0  0.0  0.0          0.0    NaN
58294  0.0  0.0  0.0          0.0    NaN


In [193]:
inf_mask  = np.isinf(new_feats['gsr_over_confounders'])
nan_mask  = new_feats['gsr_over_confounders'].isna()

print("inf rows by label:")
print(df[inf_mask]['label'].value_counts())
print("\nNaN rows by label:")
print(df[nan_mask]['label'].value_counts())

inf rows by label:
label
GSR        107987
Non_GSR       196
Name: count, dtype: int64

NaN rows by label:
label
Non_GSR    191
Name: count, dtype: int64


_Important to note that 196 of the 'inf' are Non-GSR (see above)_

In [194]:
# Current finite max
finite_max_val = new_feats['gsr_over_confounders'][np.isfinite(new_feats['gsr_over_confounders'])].max()
finite_max_val

np.float64(11977.243236206066)

In [195]:
def safe_divide_with_eps(n, d):
    e = 1e-6
    safe_d = np.where(d == 0, e, d)
    return n / safe_d

In [196]:
gsr = df['pb'] + df['sb']
confounders = df['ca'] + df['si'] + df['al'] + df['fe'] + df['ti'] + df['zn'] + df['cu']
new_feats['gsr_over_confounders'] = safe_divide_with_eps(gsr.values, confounders.values)

# check for any 'inf' or 'nan'
any(np.isinf(new_feats['gsr_over_confounders']) | new_feats['gsr_over_confounders'].isna())

False

In [197]:
# Max val w/ epsilon approach
eps_max_val = new_feats['gsr_over_confounders'].max()
eps_max_val

np.float64(100000003.81469727)

__Again, epsilon approach inflates too high. The 196 Non-GSR infinity values will inflate.__

-----
__Will try sentinel value of `-1`__

Alternative approach is to use a sentinel value that signals `denominator=0` to be deemed undefined. For example, when __d=0__ replace it with __-1__ so that any `inf` scenario outputs a negative quotient. The hope would be that a tree model like xgboost would identify the threshold between positive and negative numbers, interpreting positive quotients as reliable signals and negative quotients as unreliable and instead would rely on other features. This also address the `NaN` values, since `0 / (-1)` is 0, which is an appropriate value since all 191 NaN rows are Non-GSR.

In [198]:
def sentinel_divide(n, d):
    sentinel = -1
    safe_d = np.where(d == 0, sentinel, d)
    return n / safe_d

In [199]:
gsr = df['pb'] + df['sb']
confounders = df['ca'] + df['si'] + df['al'] + df['fe'] + df['ti'] + df['zn'] + df['cu']
new_feats['gsr_over_confounders'] = sentinel_divide(gsr.values, confounders.values)

# check for any 'inf' or 'nan'
any(np.isinf(new_feats['gsr_over_confounders']) | new_feats['gsr_over_confounders'].isna())

False

In [200]:
new_feats['gsr_over_confounders'].max() == finite_max_val

np.True_

In [201]:
new_feats['gsr_over_confounders'].min()

np.float64(-100.00000381469727)

In [202]:
# estimate pr-auc
print(f"PR-AUC estimate for gsr_over_confounders: {compute_prauc(new_feats['gsr_over_confounders'].values)}")

PR-AUC estimate for gsr_over_confounders: 0.943


__Important Follow-up__

Need to confirm that the model is correctly ignoring negative numbers (deemed undefined / unreliable). If that is true, this feature may prove useful for separating GSR from confounders.

-----
## Final Features to Consider

In [203]:
final_feats = pd.DataFrame({
    'NewFeature': new_feats.columns,
    'PR-AUC estimate': [compute_prauc(new_feats[col].values) for col in new_feats.columns]
})

final_feats.sort_values('PR-AUC estimate', ascending=False).reset_index(drop=True).rename(lambda x: x + 1).style.set_properties(subset=['NewFeature'], **{'text-align': 'left'})


,NewFeature,PR-AUC estimate
1,pb_sb_over_non_ba_o_mass,0.998800
2,pb_sb_over_non_ba_mass,0.997900
3,core_gsr_count,0.997600
4,log_pb_plus_sb,0.997300
5,gsr_over_confounders,0.943000
6,pb_times_sb,0.832300
7,pb_ba_over_non_sb_o_mass,0.773600
8,pb_ba_over_non_sb_mass,0.692500
9,ba_sb_over_non_pb_o_mass,0.691400
10,ba_sb_over_non_pb_mass,0.617200
